# browser-ai: SmolLM2 Personality Trainer

Fine-tune SmolLM2-360M-Instruct on any text to create a unique AI personality.
Uses Unsloth for fast LoRA training on Google Colab's free T4 GPU.

**Workflow:**
1. Run the first cell to install dependencies (then restart runtime manually)
2. Clone the repo
3. Upload your training text (TXT file)
4. Run training (~5-10 minutes on T4) + ONNX export
5. Download the ONNX ZIP (or fallback PyTorch ZIP if ONNX fails)
6. Open browser-ai Chat page → Upload ZIP → Chat!

All inference runs in your browser — no server needed.

In [ ]:
# Install dependencies
# Upgrade torch so optimum-cli ONNX export works
# After this finishes, go to Runtime -> Restart runtime, then continue
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth trl accelerate transformers datasets
!pip install optimum[onnx]

In [ ]:
# Clone the browser-ai repo (idempotent — uses fixed absolute path)
import os
REPO_DIR = "/content/browser-ai"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/batraaryan03/browser-ai.git "{REPO_DIR}"
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Upload your training text file
from google.colab import files
print("Upload your .txt file (50K+ characters recommended):")
uploaded = files.upload()

# Get the filename
data_file = list(uploaded.keys())[0]
print(f"Using: {data_file}")

In [ ]:
# Run training + ONNX export
!python train/smol_lora_train.py --data "{data_file}" --steps 60 --batch-size 2 --lora-rank 16 --output ./output --export-onnx

In [ ]:
# Download the model files
# First tries to download the ONNX ZIP. If that failed (e.g., torch version issue),
# falls back to downloading the PyTorch merged model directory as a ZIP.

from google.colab import files
import os
import glob
import zipfile

# Try ONNX ZIP first
onnx_zips = glob.glob("./output/personality-onnx.zip")
if onnx_zips:
    print("Downloading ONNX model ZIP...")
    files.download(onnx_zips[0])
    print("Done! ONNX model ZIP downloaded.")
else:
    # Check if ONNX directory exists but ZIP wasn't created
    onnx_dir = "./output/personality-onnx"
    if os.path.exists(onnx_dir) and os.listdir(onnx_dir):
        # Manually zip the ONNX directory
        zip_path = "./output/personality-onnx.zip"
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
            for root, dirs, files_in_dir in os.walk(onnx_dir):
                for f in files_in_dir:
                    file_path = os.path.join(root, f)
                    arcname = os.path.relpath(file_path, onnx_dir)
                    zf.write(file_path, arcname)
        print(f"Created ONNX ZIP manually: {zip_path}")
        files.download(zip_path)
        print("Done! ONNX model ZIP downloaded.")
    else:
        # Fallback: zip and download the PyTorch merged model
        print("ONNX export not available. Falling back to PyTorch model ZIP.")
        model_dir = "./output/personality"
        zip_path = "./output/personality-model.zip"
        if os.path.exists(model_dir):
            with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
                for root, dirs, files_in_dir in os.walk(model_dir):
                    for f in files_in_dir:
                        file_path = os.path.join(root, f)
                        arcname = os.path.relpath(file_path, model_dir)
                        zf.write(file_path, arcname)
            print(f"Downloading {zip_path}...")
            files.download(zip_path)
            print("Done! PyTorch model ZIP downloaded.")
            print()
            print("Note: The ONNX export failed. To use this model in the browser,")
            print("you need to convert it to ONNX first:")
            print("  1. Unzip the downloaded file")
            print("  2. Run: optimum-cli export onnx --model ./personality --task text-generation-with-past ./personality-onnx")
            print("  3. Zip the personality-onnx/ folder and upload to the Chat page")
        else:
            print("No model files found in ./output/")
            print("Available files:")
            !ls -la ./output/

# Download metadata
metadata_path = "./output/metadata.json"
if os.path.exists(metadata_path):
    files.download(metadata_path)
    print("Downloaded metadata.json")
else:
    print("metadata.json not found (optional)")

## Next Steps

### If ONNX export succeeded:
1. **Open** the browser-ai Chat page (https://browser-ai.vercel.app/models/chat)
2. **Upload** the `personality-onnx.zip` file you just downloaded
3. **Chat** with your personality! All inference runs in your browser — no server needed.

### If ONNX export failed (PyTorch fallback):
1. Unzip the downloaded `personality-model.zip` on your machine
2. Install optimum: `pip install optimum[onnx]`
3. Export to ONNX: `optimum-cli export onnx --model ./personality --task text-generation-with-past ./personality-onnx`
4. Zip the `personality-onnx/` folder
5. Upload to the Chat page

Share your ZIP with others so they can upload it to the Chat page
and experience the same personality.